In [0]:
# 06_qualidade_dados.py
from pyspark.sql import functions as F

df_bronze = spark.table("bronze.sop_forecast.vendas_raw")
df_silver = spark.table("silver.sop_forecast.vendas_limpo")

print("="*60)
print("1. COMPLETUDE — valores nulos por coluna (Bronze)")
print("="*60)
completude = df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_bronze.columns
])
display(completude)

print("="*60)
print("2. UNICIDADE — duplicatas por (date, store, item)")
print("="*60)
duplicatas = (
    df_bronze.groupBy("date", "store", "item")
    .count()
    .filter("count > 1")
)
n_duplicatas = duplicatas.count()
print(f"Registros duplicados encontrados: {n_duplicatas}")
if n_duplicatas > 0:
    display(duplicatas)

print("="*60)
print("3. CONSISTÊNCIA — domínio de store e item")
print("="*60)
consistencia = df_bronze.select(
    F.min("store").alias("min_store"), F.max("store").alias("max_store"),
    F.countDistinct("store").alias("lojas_distintas"),
    F.min("item").alias("min_item"), F.max("item").alias("max_item"),
    F.countDistinct("item").alias("itens_distintos"),
    F.min("date").alias("data_min"), F.max("date").alias("data_max")
)
display(consistencia)

print("="*60)
print("4. ACURÁCIA — valores de vendas fora do esperado (negativos)")
print("="*60)
vendas_invalidas = df_bronze.filter(F.col("sales") < 0)
print(f"Registros com vendas negativas: {vendas_invalidas.count()}")

print("="*60)
print("5. OUTLIERS — vendas acima do percentil 99.9")
print("="*60)
p999 = df_bronze.approxQuantile("sales", [0.999], 0.01)[0]
outliers = df_bronze.filter(F.col("sales") > p999)
print(f"Limite p99.9 de vendas: {p999}")
print(f"Registros acima do limite: {outliers.count()} ({outliers.count()/df_bronze.count()*100:.3f}% do total)")
display(outliers.orderBy(F.col("sales").desc()).limit(10))

print("="*60)
print("6. COMPLETUDE ESTRUTURAL — combinações loja×item×dia esperadas")
print("="*60)
n_lojas = df_bronze.select("store").distinct().count()
n_itens = df_bronze.select("item").distinct().count()
n_dias  = df_bronze.select("date").distinct().count()
esperado = n_lojas * n_itens * n_dias
real     = df_bronze.count()
print(f"Lojas × Itens × Dias esperado : {n_lojas} × {n_itens} × {n_dias} = {esperado:,}")
print(f"Registros reais               : {real:,}")
print(f"Gap                           : {esperado - real:,} ({'✅ sem gaps' if esperado == real else '⚠ há gaps temporais'})")

print("="*60)
print("RESUMO — Bronze vs. Silver")
print("="*60)
print(f"Linhas Bronze : {df_bronze.count():,}")
print(f"Linhas Silver : {df_silver.count():,}")
print(f"Linhas removidas na limpeza: {df_bronze.count() - df_silver.count():,}")